# exp005: EfficientNet V2-S SED Submission (Phase 3: Pseudo Label Retrain)

train_audio + train_soundscapes(疑似ラベル) で再学習した EfficientNet V2-S SED モデルで test_soundscapes を推論。

**入力 Dataset**: `exp005-pseudo-label-retrain` (best_fold0.pth)

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import timm
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Paths ──────────────────────────────────────────────────
_s = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
COMP_DIR = os.path.dirname(_s[0]) if _s else '/kaggle/input/birdclef-2026'
SAMPLE_SUB_CSV = f'{COMP_DIR}/sample_submission.csv'
TEST_AUDIO_DIR = f'{COMP_DIR}/test_soundscapes'

# Weight file
WEIGHT_PATH = '/kaggle/input/datasets/maekeso/exp005-pseudo-label-retrain/weights/best_fold0.pth'

for label, path in [('SAMPLE_SUB_CSV', SAMPLE_SUB_CSV), ('WEIGHT_PATH', WEIGHT_PATH), ('TEST_AUDIO_DIR', TEST_AUDIO_DIR)]:
    print(f'  {"OK" if os.path.exists(path) else "NG"} {label}: {path}')

sub_df = pd.read_csv(SAMPLE_SUB_CSV)
LABELS = [c for c in sub_df.columns if c != 'row_id']
print(f'Submission rows: {len(sub_df)}, Species: {len(LABELS)}')

In [ ]:
# ── Config (must match training) ──────────────────────────
CFG = dict(
    sample_rate  = 32000,
    n_mels       = 128,
    n_fft        = 1024,
    hop_length   = 320,
    fmin         = 20,
    fmax         = 16000,
    model_name   = 'tf_efficientnetv2_s',
    num_classes  = 234,
    segment_sec  = 5,
)

In [ ]:
# ── Mel Transform ─────────────────────────────────────────
mel_transform = nn.Sequential(
    T.MelSpectrogram(
        sample_rate=CFG['sample_rate'],
        n_fft=CFG['n_fft'],
        hop_length=CFG['hop_length'],
        n_mels=CFG['n_mels'],
        f_min=CFG['fmin'],
        f_max=CFG['fmax'],
    ),
    T.AmplitudeToDB(top_db=80),
).to(DEVICE)

def waveform_to_spec(waveforms):
    with torch.no_grad():
        specs = mel_transform(waveforms)
    specs = specs - specs.amin(dim=(-2, -1), keepdim=True)
    specs = specs / (specs.amax(dim=(-2, -1), keepdim=True) + 1e-8)
    return specs.unsqueeze(1)

In [ ]:
# ── Model (identical to training) ─────────────────────────
class AttBlockV2(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.att = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)

    def forward(self, x):
        att = torch.softmax(torch.tanh(self.att(x)), dim=-1)
        cla = self.cla(x)
        return (att * cla).sum(dim=-1)

class BirdCLEFSED(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG['model_name'],
            pretrained=False,
            in_chans=1,
            num_classes=0,
            global_pool='',
        )
        in_features = self.backbone.num_features
        self.bn = nn.BatchNorm1d(in_features)
        self.dropout = nn.Dropout(p=0.3)
        self.att_block = AttBlockV2(in_features, CFG['num_classes'])

    def forward(self, x):
        feat = self.backbone.forward_features(x)
        feat = feat.mean(dim=2)
        feat = self.bn(feat)
        feat = self.dropout(feat)
        return self.att_block(feat)

model = BirdCLEFSED().to(DEVICE)
ckpt = torch.load(WEIGHT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded weights from epoch {ckpt["epoch"]} | AUC={ckpt["best_auc"]:.4f}')

In [ ]:
# ── Inference (5s segments) ───────────────────────────────
SR = CFG['sample_rate']
SEG_SAMPLES = CFG['segment_sec'] * SR  # 5s = 160000

def predict_soundscape(filepath):
    """Predict probabilities for each 5s segment of a soundscape file."""
    waveform, sr = torchaudio.load(filepath)
    if sr != SR:
        waveform = torchaudio.functional.resample(waveform, sr, SR)
    audio = waveform.mean(dim=0)  # mono
    duration_samples = len(audio)
    
    segments = []
    end_times = []
    for end_sample in range(SEG_SAMPLES, duration_samples + 1, SEG_SAMPLES):
        end_sec = end_sample // SR
        start = end_sample - SEG_SAMPLES
        chunk = audio[start:end_sample]  # exactly 5s
        if len(chunk) < SEG_SAMPLES:
            chunk = F.pad(chunk, (0, SEG_SAMPLES - len(chunk)))
        segments.append(chunk)
        end_times.append(end_sec)
    
    if not segments:
        return [], []
    
    # Batch inference
    batch = torch.stack(segments).to(DEVICE)
    all_probs = []
    BATCH_SIZE = 64
    with torch.no_grad():
        for i in range(0, len(batch), BATCH_SIZE):
            b = batch[i:i+BATCH_SIZE]
            specs = waveform_to_spec(b)
            logits = model(specs)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
    
    all_probs = np.concatenate(all_probs, axis=0)
    return end_times, all_probs

# Run inference on all test soundscapes
test_files = sorted(glob.glob(f'{TEST_AUDIO_DIR}/*.ogg'))
print(f'Test files: {len(test_files)}')

results = {}  # row_id -> probs
t0 = time.time()

for fpath in tqdm(test_files, desc='Inference'):
    fname = os.path.splitext(os.path.basename(fpath))[0]
    end_times, probs = predict_soundscape(fpath)
    for et, p in zip(end_times, probs):
        row_id = f'{fname}_{et}'
        results[row_id] = p

elapsed = time.time() - t0
print(f'Inference done: {len(results)} segments in {elapsed:.1f}s')

In [ ]:
# ── Build Submission ──────────────────────────────────────
for idx, row in sub_df.iterrows():
    rid = row['row_id']
    if rid in results:
        sub_df.loc[idx, LABELS] = results[rid]
    # else: keep default values from sample_submission

sub_df.to_csv('submission.csv', index=False)
print(f'Submission shape: {sub_df.shape}')
print(sub_df.head())
print(f'\nMean prediction: {sub_df[LABELS].values.mean():.6f}')
print(f'Max prediction:  {sub_df[LABELS].values.max():.6f}')